In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

Libraries imported!


In [2]:
# Load the raw dataset
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
df_clean = df.copy()

print(f"Dataset loaded!")
print(f"Shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

Dataset loaded!
Shape: 1470 rows × 35 columns


In [3]:
# Drop constant columns
constant_cols = ['EmployeeCount', 'Over18', 'StandardHours']
df_clean.drop(columns=constant_cols, inplace=True)

print(f"Dropped 3 constant columns!")
print(f"Remaining columns: {df_clean.shape[1]}")

Dropped 3 constant columns!
Remaining columns: 32


In [4]:
# Encode Attrition: Yes → 1, No → 0
df_clean['AttritionEncoded'] = df_clean['Attrition'].map({'Yes': 1, 'No': 0})

print("Attrition encoded!")
print(df_clean[['Attrition', 'AttritionEncoded']].value_counts())

Attrition encoded!
Attrition  AttritionEncoded
No         0                   1233
Yes        1                    237
Name: count, dtype: int64


In [5]:
# Create AgeGroup column
age_bins = [17, 25, 35, 45, 100]
age_labels = ['18-25', '26-35', '36-45', '46+']

df_clean['AgeGroup'] = pd.cut(df_clean['Age'], bins=age_bins, labels=age_labels)

print("AgeGroup created!")
print(df_clean['AgeGroup'].value_counts().sort_index())

AgeGroup created!
AgeGroup
18-25    123
26-35    606
36-45    468
46+      273
Name: count, dtype: int64


In [6]:
# Create TenureGroup column
tenure_bins = [-1, 2, 5, 10, 100]
tenure_labels = ['0-2 yrs', '3-5 yrs', '6-10 yrs', '10+ yrs']

df_clean['TenureGroup'] = pd.cut(df_clean['YearsAtCompany'], bins=tenure_bins, labels=tenure_labels)

print("TenureGroup created!")
print(df_clean['TenureGroup'].value_counts().sort_index())

TenureGroup created!
TenureGroup
0-2 yrs     342
3-5 yrs     434
6-10 yrs    448
10+ yrs     246
Name: count, dtype: int64


In [7]:
# Create SalaryBand column
low = df_clean['MonthlyIncome'].quantile(0.33)
high = df_clean['MonthlyIncome'].quantile(0.67)

def salary_band(income):
    if income <= low:
        return 'Low'
    elif income <= high:
        return 'Medium'
    else:
        return 'High'

df_clean['SalaryBand'] = df_clean['MonthlyIncome'].apply(salary_band)

print("SalaryBand created!")
print(df_clean['SalaryBand'].value_counts())

SalaryBand created!
SalaryBand
Medium    500
Low       485
High      485
Name: count, dtype: int64


In [8]:
# Save cleaned dataset
df_clean.to_csv('hr_cleaned.csv', index=False)

print("Cleaned dataset saved as hr_cleaned.csv!")
print(f"Final shape: {df_clean.shape[0]} rows × {df_clean.shape[1]} columns")

Cleaned dataset saved as hr_cleaned.csv!
Final shape: 1470 rows × 36 columns


In [9]:
# Setup SQLite database
conn = sqlite3.connect(':memory:')
df_clean.to_sql('hr_data', conn, index=False, if_exists='replace')

print("Database ready! Now running SQL queries...")

Database ready! Now running SQL queries...


In [10]:
# SQL Query 1 - Total employees by department
q1 = pd.read_sql_query("""
    SELECT Department, COUNT(*) AS TotalEmployees
    FROM hr_data
    GROUP BY Department
    ORDER BY TotalEmployees DESC
""", conn)

print("=== Q1: Total Employees by Department ===")
print(q1)

=== Q1: Total Employees by Department ===
               Department  TotalEmployees
0  Research & Development             961
1                   Sales             446
2         Human Resources              63


In [11]:
# SQL Query 2 - Attrition rate per department
q2 = pd.read_sql_query("""
    SELECT Department, 
           COUNT(*) AS TotalEmployees,
           SUM(AttritionEncoded) AS EmployeesLeft,
           ROUND(SUM(AttritionEncoded) * 100.0 / COUNT(*), 2) AS AttritionRate_Pct
    FROM hr_data
    GROUP BY Department
    ORDER BY AttritionRate_Pct DESC
""", conn)

print("=== Q2: Attrition Rate by Department ===")
print(q2)

=== Q2: Attrition Rate by Department ===
               Department  TotalEmployees  EmployeesLeft  AttritionRate_Pct
0                   Sales             446             92              20.63
1         Human Resources              63             12              19.05
2  Research & Development             961            133              13.84


In [12]:
# SQL Query 3 - Average salary by job role
q3 = pd.read_sql_query("""
    SELECT JobRole,
           ROUND(AVG(MonthlyIncome), 0) AS AvgSalary
    FROM hr_data
    GROUP BY JobRole
    ORDER BY AvgSalary DESC
""", conn)

print("=== Q3: Average Salary by Job Role ===")
print(q3)

=== Q3: Average Salary by Job Role ===
                     JobRole  AvgSalary
0                    Manager    17182.0
1          Research Director    16034.0
2  Healthcare Representative     7529.0
3     Manufacturing Director     7295.0
4            Sales Executive     6924.0
5            Human Resources     4236.0
6         Research Scientist     3240.0
7      Laboratory Technician     3237.0
8       Sales Representative     2626.0


In [13]:
# SQL Query 4 - Headcount by gender
q4 = pd.read_sql_query("""
    SELECT Gender,
           COUNT(*) AS TotalEmployees,
           ROUND(AVG(MonthlyIncome), 0) AS AvgSalary
    FROM hr_data
    GROUP BY Gender
""", conn)

print("=== Q4: Headcount by Gender ===")
print(q4)

=== Q4: Headcount by Gender ===
   Gender  TotalEmployees  AvgSalary
0  Female             588     6687.0
1    Male             882     6381.0


In [14]:
# SQL Query 5 - Top 5 job roles by headcount
q5 = pd.read_sql_query("""
    SELECT JobRole,
           COUNT(*) AS TotalEmployees
    FROM hr_data
    GROUP BY JobRole
    ORDER BY TotalEmployees DESC
    LIMIT 5
""", conn)

print("=== Q5: Top 5 Job Roles by Headcount ===")
print(q5)

=== Q5: Top 5 Job Roles by Headcount ===
                     JobRole  TotalEmployees
0            Sales Executive             326
1         Research Scientist             292
2      Laboratory Technician             259
3     Manufacturing Director             145
4  Healthcare Representative             131


In [15]:
print("=" * 50)
print("      TASK W1.2 COMPLETE!")
print("=" * 50)
print("""
 Constant columns dropped
 Attrition encoded (Yes→1, No→0)
 AgeGroup column created
 TenureGroup column created
 SalaryBand column created
 Cleaned dataset saved as hr_cleaned.csv
 SQL Query 1 - Employees by department
 SQL Query 2 - Attrition rate by department
 SQL Query 3 - Avg salary by job role
 SQL Query 4 - Headcount by gender
 SQL Query 5 - Top 5 job roles
""")
print("=" * 50)
print("WEEK 1 COMPLETE! Ready for Week 2!")

      TASK W1.2 COMPLETE!

 Constant columns dropped
 Attrition encoded (Yes→1, No→0)
 AgeGroup column created
 TenureGroup column created
 SalaryBand column created
 Cleaned dataset saved as hr_cleaned.csv
 SQL Query 1 - Employees by department
 SQL Query 2 - Attrition rate by department
 SQL Query 3 - Avg salary by job role
 SQL Query 4 - Headcount by gender
 SQL Query 5 - Top 5 job roles

WEEK 1 COMPLETE! Ready for Week 2!
